# 02 Exploratory Data Analysis (Features)

Uwaga: ocena niezbalansowania klas i decyzja o `WeightedRandomSampler` sa wykonane w notebooku 01,
przed train/val/test split. Tutaj robimy poglebiona analize cech i zaleznosci.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
except Exception:
    variance_inflation_factor = None

sns.set_theme(style="whitegrid")


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Could not find project root with data/ and notebooks/ directories.")


PROJECT_ROOT = find_project_root()
cfg_base = PROJECT_ROOT / "data" / "processed"

train_path = cfg_base / "train.parquet"
val_path = cfg_base / "val.parquet"
test_path = cfg_base / "test.parquet"

if not (train_path.exists() and val_path.exists() and test_path.exists()):
    raise FileNotFoundError("Processed parquet files are missing. Run notebook 01 first.")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)
eda_df = train_df.copy()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("train/val/test rows:", len(train_df), len(val_df), len(test_df))
print("EDA split used for feature analysis: train")


### 1. Basic descriptive statistics (train split)


In [ ]:
numeric_cols = eda_df.select_dtypes(include=[np.number]).columns.tolist()
summary = eda_df[numeric_cols].describe().T
missing = eda_df[numeric_cols].isna().sum().rename("missing")
inf_count = np.isinf(eda_df[numeric_cols]).sum().rename("inf")

basic_stats = summary.join(missing).join(inf_count)
display(basic_stats.head(20))
print("Numeric columns:", len(numeric_cols))


### 2. Feature distributions


In [ ]:
candidate_features = [
    "return_1",
    "return_6",
    "return_24",
    "log_return_1",
    "close_open_pct",
    "high_low_pct",
    "rolling_vol_24",
    "volume_zscore_24",
    "future_return",
]
plot_cols = [c for c in candidate_features if c in eda_df.columns]

if not plot_cols:
    raise ValueError("No expected feature columns found for distribution plots.")

n_cols = 3
n_rows = int(np.ceil(len(plot_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(plot_cols):
    sns.histplot(eda_df[col].replace([np.inf, -np.inf], np.nan).dropna(), bins=60, kde=True, ax=axes[i], color="#4C78A8")
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Feature distributions (train)", y=1.02)
plt.tight_layout()
plt.show()


### 3. Correlation analysis


In [ ]:
numeric_df = eda_df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)

if "future_return" in numeric_df.columns:
    top_corr_cols = (
        numeric_df.corr(numeric_only=True)["future_return"]
        .abs()
        .sort_values(ascending=False)
        .head(12)
        .index
        .tolist()
    )
else:
    top_corr_cols = numeric_df.columns[:12].tolist()

corr_mat = numeric_df[top_corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_mat, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation heatmap (top features)")
plt.tight_layout()
plt.show()


### 4. Feature behaviour by class


In [ ]:
if "target" not in eda_df.columns:
    raise ValueError("Column 'target' is missing in train split.")

box_candidates = [
    "return_1",
    "return_6",
    "close_open_pct",
    "high_low_pct",
    "rolling_vol_24",
    "volume_zscore_24",
]
box_cols = [c for c in box_candidates if c in eda_df.columns]

if box_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(box_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.8 * n_rows))
    axes = np.array(axes).reshape(-1)

    for i, col in enumerate(box_cols):
        sns.boxplot(data=eda_df, x="target", y=col, ax=axes[i], color="#72B7B2", showfliers=False)
        axes[i].set_title(f"{col} by target")
        axes[i].set_xlabel("target (0=sell, 1=hold, 2=buy)")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No boxplot candidates found in current train split.")


### 5. Temporal patterns


In [ ]:
tmp = eda_df.copy()
if not isinstance(tmp.index, pd.DatetimeIndex):
    try:
        tmp.index = pd.to_datetime(tmp.index, utc=True, errors="coerce")
    except Exception:
        pass

if isinstance(tmp.index, pd.DatetimeIndex):
    monthly_mean = tmp["future_return"].resample("M").mean()
    plt.figure(figsize=(12, 4))
    monthly_mean.plot(color="#F58518")
    plt.title("Monthly mean of future_return (train)")
    plt.xlabel("month")
    plt.ylabel("mean future_return")
    plt.tight_layout()
    plt.show()

    hour_target_share = (
        tmp.assign(hour=tmp.index.hour)
        .groupby(["hour", "target"])
        .size()
        .rename("count")
        .reset_index()
    )
    hour_target_share["share"] = hour_target_share.groupby("hour")["count"].transform(lambda s: s / s.sum())

    plt.figure(figsize=(10, 4))
    sns.lineplot(data=hour_target_share, x="hour", y="share", hue="target", marker="o")
    plt.title("Hourly class share in train split")
    plt.ylabel("share")
    plt.tight_layout()
    plt.show()
else:
    print("Datetime index is not available. Temporal plots skipped.")


### 6. Multicollinearity (VIF)


In [ ]:
if variance_inflation_factor is None:
    print("statsmodels is not available, so VIF cannot be computed.")
else:
    vif_candidates = [
        "return_1",
        "return_6",
        "return_24",
        "log_return_1",
        "close_open_pct",
        "high_low_pct",
        "rolling_vol_24",
        "volume_zscore_24",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
    ]
    vif_cols = [c for c in vif_candidates if c in eda_df.columns]
    X = eda_df[vif_cols].replace([np.inf, -np.inf], np.nan).dropna()

    if len(vif_cols) < 2 or len(X) == 0:
        print("Not enough data/columns to compute VIF.")
    else:
        vif_rows = []
        for i, col in enumerate(X.columns):
            vif_rows.append((col, variance_inflation_factor(X.values, i)))

        vif_df = pd.DataFrame(vif_rows, columns=["feature", "VIF"]).sort_values("VIF", ascending=False)
        display(vif_df)
